# Differentiate QuantLib's original Heston engine

This notebook uses the original `FdHestonVanillaEngine`, `HestonProcess`, `HestonModel`, variance mesher, finite-difference operator, exercise events and spline readout compiled with QuantLibTorch's differentiable scalar. It does not replace the pricing algorithm with a separate Heston PDE solver.

Run from an environment where QuantLibTorch has been built and installed. All tensors in this notebook use CPU float64.

In [1]:
from datetime import date
import torch
from quantlibtorch import (
    HestonModel, heston_mesh, fd_heston, FdHestonEngine,
    PricingLayer, VanillaOption, Exercise, EquityMarket,
)
torch.set_num_threads(1)
reference = dict(v0=.045, kappa=1.5, theta=.04, sigma=.3, rho=-.6)
start, mid, end = date(2025, 1, 1), date(2025, 7, 1), date(2026, 1, 1)

## Prepare fixed numerical geometry

The original automatic mesh depends on the model and market. Prepare it once with ordinary reference numbers, then keep it fixed while differentiating or optimizing. The resulting derivatives describe changes to the pricing equations and initial state on this geometry; they exclude sensitivity to automatic mesh construction. Reassess domain and resolution adequacy when parameters move materially.

The modest grid below makes the example quick. It is not an accuracy recommendation for every contract.

In [2]:
mesh = heston_mesh(**reference, time_steps=8, x_grid=15, v_grid=9)
model = HestonModel(**reference)
option = VanillaOption(100., end)
market = EquityMarket(spot=100., valuation_date=start, rate=.03, dividend_yield=.01)
engine = FdHestonEngine(mesh=mesh, time_steps=8, mode='adjoint')
layer = PricingLayer(option, market, engine, model=model)
price = layer()
gradients = torch.autograd.grad(price, tuple(model.parameters()))
print('European call:', price.item())
for (name, _), gradient in zip(model.named_parameters(), gradients):
    print(f'dPrice/d{name}: {gradient.item():.8f}')

European call: 9.013334402550806
dPrice/dv0: 50.20674083
dPrice/dkappa: 0.06759324
dPrice/dtheta: 48.91832217
dPrice/dsigma: -2.08738547
dPrice/drho: 0.23388402


## Financial inputs stay connected

Spot, continuously compounded rate and yield can be supplied as fresh tensors on each call. Dated discount curves are also supported through `(date, discount_factor)` pairs; their initial discount must be a fixed one. An active flat rate shadowed by a dated curve is rejected.

`HestonModel` stores ordinary unconstrained parameters. For optimization, use a parameterization that keeps variance, mean reversion, long-run variance and volatility of volatility positive and correlation strictly between -1 and 1. The engine checks this domain at each call.

In [3]:
spot = torch.tensor(100., dtype=torch.float64, requires_grad=True)
rate = torch.tensor(.03, dtype=torch.float64, requires_grad=True)
price = layer(spot=spot, rate=rate)
delta, rho = torch.autograd.grad(price, (spot, rate))
print('Price, spot sensitivity, rate sensitivity:', price.item(), delta.item(), rho.item())

Price, spot sensitivity, rate sensitivity: 9.013334402550806 0.647974752157412 55.52055370793543


## Bermudan exercise and dated cash dividends

The same engine retains QuantLib's exercise projections and spot-dividend jumps. Prepare geometry using the intended financial configuration, including the dividend schedule. Dates, exercise rules, strike and cash amounts are fixed configuration in this interface. At exercise boundaries, the selected numerical branch determines the derivative.

In [4]:
cash = ((mid, 1.25),)
bermudan_mesh = heston_mesh(**reference, option_type='put', exercise='bermudan',
    exercise_dates=(mid, end), cash_dividends=cash, time_steps=8, x_grid=15, v_grid=9)
bermudan = PricingLayer(
    VanillaOption(100., end, option_type='put', exercise=Exercise.bermudan((mid, end))),
    EquityMarket(cash_dividends=cash),
    FdHestonEngine(mesh=bermudan_mesh, time_steps=8, mode='adjoint'), model=model)
value = bermudan()
print('Bermudan put with cash dividend:', value.item())
print('Initial variance sensitivity:', torch.autograd.grad(value, model.v0)[0].item())

Bermudan put with cash dividend: 7.536478325833453


Initial variance sensitivity: 48.88221016290321


## Full AD and higher derivatives

Equation adjoints provide first-order reverse derivatives. Use full AD when the loss needs derivatives of gradients. Both modes use the same original financial engine and fixed geometry. The original Heston scalar engine currently runs on CPU and prices one contract at a time; the separate latent and multi-asset APIs retain their existing batching and CUDA support.

In [5]:
full = PricingLayer(option, market,
    FdHestonEngine(mesh=mesh, time_steps=8, mode='full'), model=model)
value = full(spot=spot)
delta = torch.autograd.grad(value, spot, create_graph=True)[0]
gamma = torch.autograd.grad(delta, spot)[0]
print('Full-AD spot delta and gamma:', delta.item(), gamma.item())

Full-AD spot delta and gamma: 0.647974752157412 0.022107243603469683


## Check sensitivities using the same mesh

A centered parameter bump must reuse the prepared mesh to check the equation sensitivity. Rebuilding the mesh at each bump answers a different question. The repository tests additionally compare prices with an isolated stock QuantLib 1.40 engine and sensitivities with its original Heston solver on the same passive geometry.

In [6]:
h = 1e-5
base = tuple(parameter.detach() for parameter in model())
with torch.no_grad():
    plus = fd_heston(base[0] + h, *base[1:], mesh=mesh, time_steps=8)
    minus = fd_heston(base[0] - h, *base[1:], mesh=mesh, time_steps=8)
value = layer()
ad = torch.autograd.grad(value, model.v0)[0]
print('AD and centered fixed-mesh bump:', ad.item(), ((plus-minus)/(2*h)).item())

AD and centered fixed-mesh bump: 50.20674082650954 50.20674092470755
